# Prompt generation: Counterfactual reasoning

A walk-through for CausalARC prompt generation.

Jacqueline Maasch | August 2025

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns
import json
import platform
import ast
from itertools import permutations,product
from ast import literal_eval
import os
from os import listdir
from os.path import isfile, join

# Custom modules.
os.chdir("../causal_arc")
from carc import CausalARC
from carc_utils import UtilsARC
from carc_augment import AugmentARC
from carc_tasks_logical import TaskLogical
from carc_tasks_extension import TaskExtension
from carc_tasks_order import TaskOrder
from carc_tasks_counting import TaskCounting
from carc_tasks_sprites import TaskSprites

# View versioning.
print("python version     :", platform.python_version())
print("numpy version      :", np.__version__)
print("pandas version     :", pd.__version__)
print("matplotlib version :", matplotlib.__version__)
print("seaborn version    :", sns.__version__)

python version     : 3.12.2
numpy version      : 1.26.4
pandas version     : 2.2.3
matplotlib version : 3.10.0
seaborn version    : 0.13.2


In [2]:
c = CausalARC()
u = UtilsARC()
a = AugmentARC()
tl = TaskLogical()
te = TaskExtension()
to = TaskOrder()
tc = TaskCounting()
ts = TaskSprites()
all_tasks_dict = dict()
ttt_dict = dict()

# Define functions

In [3]:
def get_prompt_replicates(sample_dict: dict,
                          n_prompts: int = 5,
                          n_examples_in_context: int = 4) -> dict:

    # Get prompt replicates.
    l1_dict = dict()
    l3_dict = dict()
    for i in range(n_prompts):
        
        # Get L1 prompt.
        l1_prompt_dict = c.get_prompt(sample_dict, 
                                      n_examples = n_examples_in_context-1,
                                      counterfactuals = False,
                                      scm = False,
                                      n_counterfactuals = 0, 
                                      problem_type = "cf_reasoning")
        l1_dict[f"Replicate {i}"] = l1_prompt_dict

        # Get L3 prompt.
        l3_prompt_dict = c.get_prompt(sample_dict, 
                                      n_examples = n_examples_in_context//2,
                                      counterfactuals = True,
                                      scm = False,
                                      n_counterfactuals = 1, 
                                      problem_type = "cf_reasoning")
        l3_dict[f"Replicate {i}"] = l3_prompt_dict

    return {"L1": l1_dict, "L3": l3_dict}

# Get prompts

In [4]:
n_examples = 5
n_examples_in_context = 4
n_prompts = 5
scms = ["SCMffb8", "SCMtzlq", 
        "SCMz750", "SCMwoev", 
        "SCMfuy3", "SCMm5ob"]
themes = ["ordering", "ordering", 
          "extending", "extending", 
          "counting", "counting"]
methods = [to.task_SCMffb8, to.task_SCMtzlq, 
           te.task_SCMz750, te.task_SCMwoev, 
           tc.task_SCMfuy3, tc.task_SCMm5ob]
size = (25,25)

no_size = ["SCMfuy3", "SCMffb8"]

task_names = []
for i in range(len(methods)):

    task_name = scms[i]
    task_names.append(task_name)
    print(f"\n-*- {task_name} -*-")

    # Get sample dictionary.
    if task_name not in no_size:
        sample_dict = methods[i](size = size,
                                 n_examples = n_examples, # Total input-output pairs per sample.
                                 plot = False,
                                 plot_type = "input_output", # "single"
                                 figsize = (5,2),
                                 grid = True)
    else:
        sample_dict = methods[i](n_examples = n_examples, # Total input-output pairs per sample.
                                 plot = False,
                                 plot_type = "input_output", # "single"
                                 figsize = (5,2),
                                 grid = True)
    ttt_dict[task_name] = sample_dict

    # Get prompt replicates.
    prompt_replicates = get_prompt_replicates(sample_dict,
                                              n_prompts = n_prompts,
                                              n_examples_in_context = n_examples_in_context)
    all_tasks_dict[task_name] = prompt_replicates

    print("\n\nL1 prompt")
    print(prompt_replicates["L1"]["Replicate 0"])

    print("\n\nL3 prompt")
    print(prompt_replicates["L3"]["Replicate 0"])

print("\nTotal tasks:", len(task_names))


-*- SCMffb8 -*-


L1 prompt
{'prompt': 'You must solve the following puzzle by discovering the deterministic rule that maps inputs to outputs. You will then be asked to predict the output for a counterfactual example. Both the inputs and outputs are 2D grids of colored pixels. We provide example input-output pairs as demonstration. Grids are provided as Python arrays. You must output only a single Python array, and do not explain your reasoning.\nExample input-output arrays:\n[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 5, 0, 0], [0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 5, 0, 0], [0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 5, 0, 5, 0, 0], [0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 

# Export full JSON

In [5]:
with open("../data/cf_reasoning_not_logical/cf_reasoning_not_logical_api.json", "w") as f:
    json.dump(all_tasks_dict, f, indent = 4) # indent for readability.

In [7]:
with open('../data/cf_reasoning_not_logical/cf_reasoning_not_logical_ttt.json', 'w') as f:
    json.dump(ttt_dict, f, indent = 4) # indent for readability.

In [8]:
# Export solutions file.
ttt_dict_solutions = dict()
for task,d in ttt_dict.items():
    test_output = d["test"][0].get("output")
    ttt_dict_solutions[task] = [test_output]

with open('../data/cf_reasoning_not_logical/cf_reasoning_not_logical_ttt_solutions.json', 'w') as f:
    json.dump(ttt_dict_solutions, f, indent = 4) # indent for readability.

# End of document